# Modélisation – Expérience de robustesse sans la variable `conso_mixte`

## Objectif de la modélisation

À l'issue des étapes d'exploration, de visualisation et de prétraitement des données, le jeu de données est désormais prêt à être exploité dans le cadre de la phase de modélisation.

L'objectif de cette étape est de construire un ou plusieurs modèles de régression capables de prédire les émissions de dioxyde de carbone (CO₂) des véhicules à partir de leurs caractéristiques techniques et énergétiques.

Plusieurs modèles de régression seront entraînés puis comparés afin d'identifier celui offrant les meilleures performances prédictives. L'évaluation reposera principalement sur les métriques MAE (Mean Absolute Error), RMSE (Root Mean Squared Error) et R² (coefficient de détermination). La MAPE (Mean Absolute Percentage Error) pourra également être utilisée à titre complémentaire.

Conformément aux objectifs du projet, une attention particulière sera portée aux performances des modèles. Leur interprétabilité sera également prise en compte afin de faciliter l'analyse des résultats et la compréhension de l'influence des variables explicatives sur les émissions de CO₂.

## Expérience : suppression de la variable `conso_mixte`

Une seconde expérimentation est réalisée afin d'évaluer la robustesse des modèles de régression. La variable `conso_mixte`, très fortement corrélée aux émissions de CO₂, est volontairement retirée des variables explicatives.

L'objectif est d'analyser l'impact de cette suppression sur les performances des différents modèles et de déterminer leur capacité à prédire les émissions de CO₂ à partir des seules caractéristiques techniques des véhicules.

In [38]:
# Import des bibliothèques

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from sklearn.tree import DecisionTreeRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error
)

## Chargement des jeux de données prétraités

Les jeux de données préparés lors de la phase de prétraitement sont chargés afin d'être utilisés pour l'entraînement et l'évaluation des différents modèles de régression.

In [2]:
# Chargement des jeux de données

X_train = pd.read_csv("../data/X_train_final.csv")
X_test = pd.read_csv("../data/X_test_final.csv")

y_train = pd.read_csv("../data/y_train.csv")["co2"]
y_test = pd.read_csv("../data/y_test.csv")["co2"]

In [3]:
print("X_train :", X_train.shape)
print("X_test :", X_test.shape)
print("y_train :", y_train.shape)
print("y_test :", y_test.shape)

X_train : (44002, 93)
X_test : (11001, 93)
y_train : (44002,)
y_test : (11001,)


In [4]:
# Suppression de la variable conso_mixte afin d'évaluer la robustesse des modèles

X_train_sans_conso = X_train.drop(columns=["conso_mixte"])
X_test_sans_conso = X_test.drop(columns=["conso_mixte"])

print("Nouvelle dimension de X_train :", X_train_sans_conso.shape)
print("Nouvelle dimension de X_test :", X_test_sans_conso.shape)

Nouvelle dimension de X_train : (44002, 92)
Nouvelle dimension de X_test : (11001, 92)


## Vérification des jeux de données

Avant de débuter l'entraînement des modèles, une vérification des dimensions des jeux de données est réalisée afin de s'assurer de leur cohérence.

In [5]:
print("Dimensions de X_train :", X_train_sans_conso.shape)
print("Dimensions de X_test :", X_test_sans_conso.shape)

print("Dimensions de y_train :", y_train.shape)
print("Dimensions de y_test :", y_test.shape)

Dimensions de X_train : (44002, 92)
Dimensions de X_test : (11001, 92)
Dimensions de y_train : (44002,)
Dimensions de y_test : (11001,)


In [6]:
X_train_sans_conso.head()

,puiss_max,co_typ_1,nox,ptcl,masse_ordma_min,masse_ordma_max,rapport_puissance_masse,lib_mrq_ASTON MARTIN,lib_mrq_AUDI,lib_mrq_BENTLEY,...,typ_boite_nb_rapp_A 9,typ_boite_nb_rapp_D 5,typ_boite_nb_rapp_D 6,typ_boite_nb_rapp_D 7,typ_boite_nb_rapp_M 5,typ_boite_nb_rapp_M 6,typ_boite_nb_rapp_M 7,typ_boite_nb_rapp_S 6,typ_boite_nb_rapp_V .,typ_boite_nb_rapp_V 0
0,120.0,0.046,0.197,0.000,2076,2185,0.057803,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
1,125.0,0.440,0.030,0.001,1380,1380,0.090580,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,120.0,0.401,0.213,0.001,2246,2709,0.053428,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,120.0,0.175,0.226,0.001,2186,2355,0.054895,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
4,120.0,0.046,0.197,0.000,2186,2350,0.054895,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0



Les jeux de données utilisés pour la modélisation sont désormais entièrement numériques et prêts à être exploités par les différents algorithmes de Machine Learning. Dans cette expérimentation, la variable `conso_mixte` a été supprimée, ce qui porte le nombre de variables explicatives à **92**.

Le jeu d'entraînement (`X_train_sans_conso`) est utilisé pour apprendre les relations entre les variables explicatives et les émissions de CO₂, tandis que le jeu de test (`X_test_sans_conso`) permet d'évaluer les performances des modèles sur des données jamais vues pendant l'entraînement.

Les variables catégorielles ont été transformées par encodage One-Hot lors de la phase de prétraitement, ce qui permet à l'ensemble des modèles de travailler exclusivement sur des variables numériques.

In [7]:
X_train_sans_conso.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44002 entries, 0 to 44001
Data columns (total 92 columns):
 #   Column                           Non-Null Count  Dtype  
---  ------                           --------------  -----  
 0   puiss_max                        44002 non-null  float64
 1   co_typ_1                         44002 non-null  float64
 2   nox                              44002 non-null  float64
 3   ptcl                             44002 non-null  float64
 4   masse_ordma_min                  44002 non-null  int64  
 5   masse_ordma_max                  44002 non-null  int64  
 6   rapport_puissance_masse          44002 non-null  float64
 7   lib_mrq_ASTON MARTIN             44002 non-null  float64
 8   lib_mrq_AUDI                     44002 non-null  float64
 9   lib_mrq_BENTLEY                  44002 non-null  float64
 10  lib_mrq_BMW                      44002 non-null  float64
 11  lib_mrq_CADILLAC                 44002 non-null  float64
 12  lib_mrq_CHEVROLET 

In [8]:
X_train_sans_conso.describe()

,puiss_max,co_typ_1,nox,ptcl,masse_ordma_min,masse_ordma_max,rapport_puissance_masse,lib_mrq_ASTON MARTIN,lib_mrq_AUDI,lib_mrq_BENTLEY,...,typ_boite_nb_rapp_A 9,typ_boite_nb_rapp_D 5,typ_boite_nb_rapp_D 6,typ_boite_nb_rapp_D 7,typ_boite_nb_rapp_M 5,typ_boite_nb_rapp_M 6,typ_boite_nb_rapp_M 7,typ_boite_nb_rapp_S 6,typ_boite_nb_rapp_V .,typ_boite_nb_rapp_V 0
count,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,...,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000,44002.000000
mean,118.888580,0.180790,0.299221,0.000898,2103.413208,2343.264056,0.057724,0.001409,0.006113,0.000432,...,0.000591,0.001136,0.000523,0.000455,0.018181,0.578246,0.000341,0.000023,0.000045,0.005568
std,45.091507,0.144731,0.417127,0.000969,293.418196,422.748496,0.024769,0.037511,0.077949,0.020776,...,0.024301,0.033690,0.022857,0.021315,0.133607,0.493845,0.018460,0.004767,0.006742,0.074411
min,28.000000,0.005000,0.000000,0.000000,825.000000,825.000000,0.020144,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,100.000000,0.061000,0.197000,0.000000,1982.000000,2075.000000,0.045746,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,120.000000,0.137000,0.214000,0.001000,2076.000000,2355.000000,0.052016,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
75%,120.000000,0.296000,0.228000,0.001000,2246.000000,2709.000000,0.059172,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
max,585.000000,0.968000,1.846000,0.023000,2760.000000,3094.000000,0.373802,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


## Vérification finale des jeux de données

Avant d'entraîner les modèles, une dernière vérification est réalisée afin de confirmer l'absence de valeurs manquantes dans les jeux d'entraînement et de test. Cette étape garantit que les données sont prêtes pour la modélisation.

In [9]:
# Vérification finale des valeurs manquantes

print("Nombre total de NaN dans X_train :", X_train_sans_conso.isna().sum().sum())
print("Nombre total de NaN dans X_test  :", X_test_sans_conso.isna().sum().sum())

print("Nombre total de NaN dans y_train :", y_train.isna().sum())
print("Nombre total de NaN dans y_test  :", y_test.isna().sum())

Nombre total de NaN dans X_train : 0
Nombre total de NaN dans X_test  : 0
Nombre total de NaN dans y_train : 0
Nombre total de NaN dans y_test  : 0


## Régression linéaire

La régression linéaire constitue le modèle de référence (baseline) de notre étude. Son principe repose sur l'hypothèse d'une relation linéaire entre les variables explicatives et les émissions de CO₂.

Ce modèle présente plusieurs avantages : il est rapide à entraîner, simple à interpréter et permet d'identifier l'influence des différentes variables sur la variable cible. Il servira de point de comparaison pour évaluer les performances des modèles plus complexes étudiés par la suite.

### Entraînement du modèle

In [10]:
# Initialisation du modèle
reg_lin = LinearRegression()

# Entraînement
reg_lin.fit(X_train_sans_conso, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


### Prédictions

In [11]:
y_pred_lin = reg_lin.predict(X_test_sans_conso)

### Évaluation

In [12]:
mae_lin = mean_absolute_error(y_test, y_pred_lin)
rmse_lin = np.sqrt(mean_squared_error(y_test, y_pred_lin))
r2_lin = r2_score(y_test, y_pred_lin)
mape_lin = mean_absolute_percentage_error(y_test, y_pred_lin)

print(f"MAE  : {mae_lin:.2f}")
print(f"RMSE : {rmse_lin:.2f}")
print(f"R²   : {r2_lin:.4f}")
print(f"MAPE : {mape_lin:.4f}")

MAE  : 9.63
RMSE : 12.92
R²   : 0.8570
MAPE : 0.0496


### Vérification de la capacité de généralisation

In [13]:
print("Score R² sur le jeu d'entraînement :", reg_lin.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", reg_lin.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.861187441372675
Score R² sur le jeu de test        : 0.8569621436976695


Les coefficients de détermination obtenus sur les jeux d'entraînement (**R² = 0,8612**) et de test (**R² = 0,8570**) sont très proches.

Cet écart très faible montre que la régression linéaire conserve une bonne capacité de généralisation malgré la suppression de la variable `conso_mixte`. Le modèle ne présente pas de surapprentissage significatif.

### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne une dégradation importante des performances de la Régression linéaire.

Le coefficient de détermination passe de **0,9974 à 0,8570**, tandis que le RMSE augmente de **1,74 à 12,92 g/km**. La MAE atteint **9,63 g/km** et la MAPE **4,96 %**, ce qui confirme que `conso_mixte` constituait une variable particulièrement informative.

Néanmoins, le modèle conserve une capacité prédictive intéressante, avec un R² de 0,8570, montrant que les autres caractéristiques techniques des véhicules permettent encore d'expliquer près de 86 % de la variabilité des émissions de CO₂.

Cette expérimentation met ainsi en évidence la forte influence de `conso_mixte` sur les performances du modèle, sans permettre à elle seule de conclure à une fuite de données.

## Régression Ridge

La régression Ridge est une extension de la régression linéaire qui intègre une pénalisation de type L2 sur les coefficients du modèle.

Cette régularisation permet de limiter l'influence des variables fortement corrélées et de réduire le risque de surapprentissage. Elle est particulièrement adaptée lorsque le jeu de données comporte un grand nombre de variables explicatives, comme c'est le cas après l'encodage One-Hot des variables catégorielles.

L'objectif est d'évaluer si cette pénalisation améliore les performances prédictives par rapport à la régression linéaire classique.

### Entraînement du modèle

In [14]:
# Initialisation du modèle
ridge = Ridge(alpha=1.0)

# Entraînement
ridge.fit(X_train_sans_conso, y_train)

,alpha,1.0
,fit_intercept,True
,copy_X,True
,max_iter,None
,tol,0.0001
,solver,'auto'
,positive,False
,random_state,None


### Prédictions

In [15]:
# Prédictions sur le jeu de test
y_pred_ridge = ridge.predict(X_test_sans_conso)

### Évaluation

In [16]:
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = np.sqrt(mean_squared_error(y_test, y_pred_ridge))
r2_ridge = r2_score(y_test, y_pred_ridge)
mape_ridge = mean_absolute_percentage_error(y_test, y_pred_ridge)

print(f"MAE  : {mae_ridge:.2f}")
print(f"RMSE : {rmse_ridge:.2f}")
print(f"R²   : {r2_ridge:.4f}")
print(f"MAPE : {mape_ridge:.4f}")

MAE  : 9.67
RMSE : 12.91
R²   : 0.8572
MAPE : 0.0498


### Vérification de la capacité de généralisation

In [17]:
print("Score R² sur le jeu d'entraînement :", ridge.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", ridge.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.8607567113813206
Score R² sur le jeu de test        : 0.8572404894054462


La suppression de la variable `conso_mixte` entraîne également une dégradation importante des performances de la régression Ridge.

Le coefficient de détermination passe de **0,9974 à 0,8572**, tandis que le RMSE augmente de **1,74 à 12,91 g/km**. La MAE atteint **9,67 g/km** et la MAPE **4,98 %**, ce qui confirme l'influence majeure de `conso_mixte` sur les performances du modèle.

Néanmoins, le modèle conserve une capacité prédictive intéressante, avec un R² de 0,8572, montrant que les autres caractéristiques techniques des véhicules permettent encore d'expliquer près de 86 % de la variabilité des émissions de CO₂.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,8608**) et de test (**R² = 0,8572**) sont très proches, ce qui indique une bonne capacité de généralisation, sans surapprentissage significatif.

Enfin, les performances du Ridge restent très proches de celles de la régression linéaire sans `conso_mixte`. Dans cette expérimentation, la régularisation L2 n'apporte donc pas d'amélioration notable.

## Régression Lasso

La régression Lasso est une variante de la régression linéaire qui applique une pénalisation de type L1 sur les coefficients du modèle.

Contrairement à la régression Ridge, cette pénalisation peut réduire certains coefficients exactement à zéro, réalisant ainsi une sélection automatique des variables les moins pertinentes. Cette propriété permet d'obtenir des modèles plus simples et potentiellement plus interprétables.

L'objectif est d'évaluer si cette régularisation améliore les performances prédictives tout en simplifiant le modèle.

### Entraînement du modèle

In [18]:
# Initialisation du modèle
lasso = Lasso(alpha=1.0)

# Entraînement
lasso.fit(X_train_sans_conso, y_train)

,alpha,1.0
,fit_intercept,True
,precompute,False
,copy_X,True
,max_iter,1000
,tol,0.0001
,warm_start,False
,positive,False
,random_state,None
,selection,'cyclic'


### Prédictions

In [19]:
# Prédictions sur le jeu de test
y_pred_lasso = lasso.predict(X_test_sans_conso)

### Évaluation

In [20]:
mae_lasso = mean_absolute_error(y_test, y_pred_lasso)
rmse_lasso = np.sqrt(mean_squared_error(y_test, y_pred_lasso))
r2_lasso = r2_score(y_test, y_pred_lasso)
mape_lasso = mean_absolute_percentage_error(y_test, y_pred_lasso)

print(f"MAE  : {mae_lasso:.2f}")
print(f"RMSE : {rmse_lasso:.2f}")
print(f"R²   : {r2_lasso:.4f}")
print(f"MAPE : {mape_lasso:.4f}")

MAE  : 12.37
RMSE : 16.99
R²   : 0.7526
MAPE : 0.0639


### Vérification de la capacité de généralisation

In [21]:
print("Score R² sur le jeu d'entraînement :", lasso.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", lasso.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.7550290860320563
Score R² sur le jeu de test        : 0.7525719328812421


### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne également une dégradation importante des performances de la régression Lasso, qui reste le modèle linéaire le plus affecté par cette suppression.

Le coefficient de détermination passe de **0,9923 à 0,7526**, tandis que le RMSE augmente de **3,01 à 16,99 g/km**. La MAE atteint **12,37 g/km** et la MAPE **6,39 %**, ce qui confirme l'influence majeure de `conso_mixte` sur les performances du modèle.

Néanmoins, le modèle conserve une certaine capacité prédictive, avec un R² de 0,7526, montrant que les autres caractéristiques techniques des véhicules permettent encore d'expliquer environ **75 % de la variabilité des émissions de CO₂**.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,7550**) et de test (**R² = 0,7526**) sont très proches, ce qui indique une bonne capacité de généralisation, sans surapprentissage significatif.

Enfin, les performances du Lasso restent inférieures à celles de la régression linéaire et de Ridge sans `conso_mixte`. Dans cette expérimentation, la régularisation L1 n'apporte donc pas d'amélioration des performances prédictives.

## Arbre de décision

L'arbre de décision est un modèle non linéaire capable de capturer des relations complexes entre les variables explicatives et la variable cible.

Contrairement aux modèles de régression linéaire, Ridge et Lasso, il ne suppose aucune relation linéaire entre les variables. Il segmente progressivement les données en fonction des variables les plus discriminantes afin de minimiser l'erreur de prédiction.

L'objectif est d'évaluer si cette approche permet d'améliorer les performances prédictives par rapport aux modèles linéaires.

### Entraînement du modèle

In [22]:

# Initialisation du modèle
tree = DecisionTreeRegressor(random_state=42)

# Entraînement
tree.fit(X_train_sans_conso, y_train)

,criterion,'squared_error'
,splitter,'best'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


### Prédictions

In [23]:
# Prédictions sur le jeu de test
y_pred_tree = tree.predict(X_test_sans_conso)

### Évaluation

In [24]:
mae_tree = mean_absolute_error(y_test, y_pred_tree)
rmse_tree = np.sqrt(mean_squared_error(y_test, y_pred_tree))
r2_tree = r2_score(y_test, y_pred_tree)
mape_tree = mean_absolute_percentage_error(y_test, y_pred_tree)

print(f"MAE  : {mae_tree:.2f}")
print(f"RMSE : {rmse_tree:.2f}")
print(f"R²   : {r2_tree:.4f}")
print(f"MAPE : {mape_tree:.4f}")

MAE  : 3.19
RMSE : 5.40
R²   : 0.9750
MAPE : 0.0173


### Vérification de la capacité de généralisation

In [25]:
print("Score R² sur le jeu d'entraînement :", tree.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", tree.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.9851841498204048
Score R² sur le jeu de test        : 0.9749996490145112


### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne également une dégradation des performances de l'Arbre de décision, qui reste néanmoins le modèle le plus performant parmi ceux évalués jusqu'à présent dans cette expérimentation.

Le coefficient de détermination passe de **0,9991 à 0,9750**, tandis que le RMSE augmente de **1,04 à 5,40 g/km**. La MAE atteint **3,19 g/km** et la MAPE **1,73 %**, ce qui confirme l'influence importante de `conso_mixte` sur les performances du modèle.

Néanmoins, le modèle conserve une excellente capacité prédictive, avec un R² de 0,9750, montrant que les autres caractéristiques techniques des véhicules permettent encore d'expliquer environ **97,5 % de la variabilité des émissions de CO₂**.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,9852**) et de test (**R² = 0,9750**) présentent un écart d'environ 0,0102, ce qui indique un léger surapprentissage, sans remettre en cause les bonnes performances du modèle.

Enfin, l'Arbre de décision surpasse nettement les modèles linéaires évalués sans `conso_mixte`, ce qui met en évidence l'intérêt des approches non linéaires pour prédire les émissions de CO₂ à partir des caractéristiques techniques des véhicules.

### Importance des variables

In [26]:
importance = pd.DataFrame({
    "Variable": X_train_sans_conso.columns,
    "Importance": tree.feature_importances_
})

importance.sort_values("Importance", ascending=False).head(15)

,Variable,Importance
5,masse_ordma_max,0.472294
0,puiss_max,0.180695
51,cod_cbr_ES,0.121876
78,typ_boite_nb_rapp_A 5,0.069093
2,nox,0.054960
1,co_typ_1,0.036395
80,typ_boite_nb_rapp_A 7,0.018809
4,masse_ordma_min,0.011592
3,ptcl,0.008733
6,rapport_puissance_masse,0.003607


L'analyse de l'importance des variables montre qu'après la suppression de `conso_mixte`, l'Arbre de décision s'appuie principalement sur la masse maximale du véhicule (`masse_ordma_max`), avec une importance de **47,23 %**, la puissance maximale (`puiss_max`), avec **18,07 %**, et le type de carburant (`cod_cbr_ES`), avec **12,19 %**.

Le nombre de rapports de boîte de vitesses (`typ_boite_nb_rapp_A 5`) contribue également aux prédictions (**6,91 %**), ainsi que les émissions de NOx (`nox`, **5,50 %**) et de CO (`co_typ_1`, **3,64 %**). Les autres variables présentent une importance plus limitée.

Ces résultats montrent que le modèle est capable de s'appuyer sur d'autres caractéristiques techniques des véhicules pour prédire les émissions de CO₂ malgré l'absence de `conso_mixte`.

## Random Forest

Le Random Forest est un modèle d'ensemble reposant sur plusieurs arbres de décision. Contrairement à un arbre unique, il combine les prédictions de plusieurs arbres afin d'améliorer la précision et la stabilité des résultats.

Cette approche permet de capturer des relations non linéaires complexes entre les variables explicatives et les émissions de CO₂, tout en limitant le risque de surapprentissage par rapport à un arbre de décision isolé.

L'objectif est d'évaluer si cette méthode permet d'améliorer les performances prédictives obtenues avec l'Arbre de décision, malgré la suppression de la variable `conso_mixte`.


### Entraînement du modèle

In [29]:
# Initialisation du modèle
random_forest = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

# Entraînement
random_forest.fit(X_train_sans_conso, y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


### Prédictions

In [30]:
# Prédictions sur le jeu de test
y_pred_rf = random_forest.predict(X_test_sans_conso)

### Évaluation

In [31]:
mae_rf = mean_absolute_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, y_pred_rf))
r2_rf = r2_score(y_test, y_pred_rf)
mape_rf = mean_absolute_percentage_error(y_test, y_pred_rf)

print(f"MAE  : {mae_rf:.2f}")
print(f"RMSE : {rmse_rf:.2f}")
print(f"R²   : {r2_rf:.4f}")
print(f"MAPE : {mape_rf:.4f}")

MAE  : 3.17
RMSE : 4.82
R²   : 0.9801
MAPE : 0.0170


### Vérification de la capacité de généralisation

In [32]:
print("Score R² sur le jeu d'entraînement :", random_forest.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", random_forest.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.9841763417406276
Score R² sur le jeu de test        : 0.9800745986769348


### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne également une dégradation des performances du Random Forest, qui conserve néanmoins une excellente capacité prédictive.

Le coefficient de détermination passe de **0,9996 à 0,9801**, tandis que le RMSE augmente de **0,72 à 4,82 g/km**. La MAE atteint **3,17 g/km** et la MAPE **1,70 %**, ce qui confirme l'influence importante de `conso_mixte` sur les performances du modèle.

Néanmoins, le modèle conserve un R² de 0,9801, montrant que les autres caractéristiques techniques des véhicules permettent encore d'expliquer environ **98 % de la variabilité des émissions de CO₂**.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,9842**) et de test (**R² = 0,9801**) présentent un écart très faible, d'environ 0,0041, ce qui indique une bonne capacité de généralisation, sans surapprentissage significatif.

Enfin, Random Forest améliore les performances de l'Arbre de décision sans `conso_mixte`, notamment en réduisant le RMSE de **5,40 à 4,82 g/km**. Il devient ainsi le modèle le plus performant parmi ceux évalués jusqu'à présent dans cette expérimentation.

## XGBoost

XGBoost (Extreme Gradient Boosting) est un algorithme d'ensemble reposant sur une succession d'arbres de décision. Contrairement au Random Forest, qui combine plusieurs arbres entraînés indépendamment, XGBoost construit progressivement ses arbres afin de corriger les erreurs des prédictions précédentes.

Cette approche permet de capturer des relations non linéaires complexes entre les caractéristiques techniques des véhicules et leurs émissions de CO₂, tout en intégrant des mécanismes de régularisation pour limiter le surapprentissage.

L'objectif est d'évaluer si XGBoost conserve ses excellentes performances prédictives après la suppression de la variable `conso_mixte`, et de comparer ses résultats à ceux du Random Forest.


### Entraînement du modèle

In [34]:
# Initialisation du modèle
xgb_sans_conso = XGBRegressor(
    random_state=42,
    n_jobs=-1
)

# Entraînement
xgb_sans_conso.fit(X_train_sans_conso, y_train)

,objective,'reg:squarederror'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,None
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


### Prédictions

In [35]:
# Prédictions sur le jeu de test
y_pred_xgb_sans_conso = xgb_sans_conso.predict(X_test_sans_conso)

### Évaluation

In [36]:
mae_xgb_sans_conso = mean_absolute_error(y_test, y_pred_xgb_sans_conso)
rmse_xgb_sans_conso = np.sqrt(mean_squared_error(y_test, y_pred_xgb_sans_conso))
r2_xgb_sans_conso = r2_score(y_test, y_pred_xgb_sans_conso)
mape_xgb_sans_conso = mean_absolute_percentage_error(y_test, y_pred_xgb_sans_conso)

print(f"MAE  : {mae_xgb_sans_conso:.2f}")
print(f"RMSE : {rmse_xgb_sans_conso:.2f}")
print(f"R²   : {r2_xgb_sans_conso:.4f}")
print(f"MAPE : {mape_xgb_sans_conso:.4f}")

MAE  : 3.34
RMSE : 4.82
R²   : 0.9801
MAPE : 0.0181


### Vérification de la capacité de généralisation

In [37]:
print("Score R² sur le jeu d'entraînement :", xgb_sans_conso.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", xgb_sans_conso.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.9827844858485831
Score R² sur le jeu de test        : 0.98006082201631


### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne une dégradation des performances de XGBoost, qui conserve néanmoins une excellente capacité prédictive.

Le coefficient de détermination passe de **0,9997 à 0,9801**, tandis que le RMSE augmente de **0,59 à 4,82 g/km**. La MAE atteint **3,34 g/km** et la MAPE **1,81 %**, confirmant l'influence importante de `conso_mixte` sur les performances du modèle.

Malgré cette suppression, XGBoost conserve un R² de **0,9801**, ce qui signifie que les autres caractéristiques techniques des véhicules permettent encore d'expliquer environ **98 % de la variabilité des émissions de CO₂**.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,9828**) et de test (**R² = 0,9801**) présentent un écart limité à **0,0027**, ce qui témoigne d'une bonne capacité de généralisation, sans surapprentissage significatif.

Enfin, XGBoost obtient des performances très proches de celles du Random Forest, avec un RMSE de **4,82 g/km** et un R² de **0,9801** pour les deux modèles. Random Forest conserve toutefois un léger avantage sur la MAE (**3,17 contre 3,34 g/km**) et la MAPE (**1,70 % contre 1,81 %**).

Ces résultats montrent que XGBoost reste performant même en l'absence de `conso_mixte`, mais qu'il ne surpasse pas Random Forest dans cette expérimentation.

## LightGBM

LightGBM (Light Gradient Boosting Machine) est un algorithme d'ensemble basé sur le gradient boosting, comme XGBoost. Il construit successivement des arbres de décision afin de corriger les erreurs des prédictions précédentes.

Il se distingue notamment par sa rapidité d'entraînement et sa stratégie de croissance des arbres, qui lui permet de capturer efficacement des relations non linéaires complexes.

L'objectif est d'évaluer sa capacité à prédire les émissions de CO₂ sans la variable `conso_mixte`, puis de comparer ses performances à celles du Random Forest et de XGBoost.


### Entraînement du modèle

In [39]:
# Initialisation du modèle
lgbm_sans_conso = LGBMRegressor(
    random_state=42,
    verbosity=-1
)

# Entraînement
lgbm_sans_conso.fit(X_train_sans_conso, y_train)

,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


### Prédictions

In [40]:
# Prédictions sur le jeu de test
y_pred_lgbm_sans_conso = lgbm_sans_conso.predict(X_test_sans_conso)

### Évaluation

In [41]:
mae_lgbm_sans_conso = mean_absolute_error(y_test, y_pred_lgbm_sans_conso)
rmse_lgbm_sans_conso = np.sqrt(mean_squared_error(y_test, y_pred_lgbm_sans_conso))
r2_lgbm_sans_conso = r2_score(y_test, y_pred_lgbm_sans_conso)
mape_lgbm_sans_conso = mean_absolute_percentage_error(y_test, y_pred_lgbm_sans_conso)

print(f"MAE  : {mae_lgbm_sans_conso:.2f}")
print(f"RMSE : {rmse_lgbm_sans_conso:.2f}")
print(f"R²   : {r2_lgbm_sans_conso:.4f}")
print(f"MAPE : {mape_lgbm_sans_conso:.4f}")

MAE  : 3.71
RMSE : 5.44
R²   : 0.9747
MAPE : 0.0201


### Vérification de la capacité de généralisation

In [42]:
print("Score R² sur le jeu d'entraînement :", lgbm_sans_conso.score(X_train_sans_conso, y_train))
print("Score R² sur le jeu de test        :", lgbm_sans_conso.score(X_test_sans_conso, y_test))

Score R² sur le jeu d'entraînement : 0.976615602499422
Score R² sur le jeu de test        : 0.974657681648519


### Interprétation des résultats

La suppression de la variable `conso_mixte` entraîne une dégradation des performances de LightGBM, qui conserve néanmoins une bonne capacité prédictive.

Le coefficient de détermination passe de **0,9986 à 0,9747**, tandis que le RMSE augmente de **1,29 à 5,44 g/km**. La MAE atteint **3,71 g/km** et la MAPE **2,01 %**, confirmant l'influence importante de `conso_mixte` sur les performances du modèle.

Malgré cette suppression, LightGBM conserve un R² de **0,9747**, ce qui signifie que les autres caractéristiques techniques des véhicules permettent encore d'expliquer environ **97,47 % de la variabilité des émissions de CO₂**.

Les scores obtenus sur les jeux d'entraînement (**R² = 0,9766**) et de test (**R² = 0,9747**) présentent un écart limité à **0,0020**, ce qui témoigne d'une bonne capacité de généralisation, sans surapprentissage significatif.

Enfin, LightGBM obtient des performances légèrement inférieures à celles du Random Forest et de XGBoost, qui atteignent tous deux un R² de **0,9801** et un RMSE de **4,82 g/km**.

Ces résultats montrent que LightGBM reste performant même en l'absence de `conso_mixte`, mais que Random Forest et XGBoost offrent de meilleurs résultats dans cette expérimentation.

## Comparaison des modèles

Les performances des différents modèles de régression sont comparées à l'aide de quatre indicateurs : le MAE (Mean Absolute Error), le RMSE (Root Mean Squared Error), le coefficient de détermination (R²) et le MAPE (Mean Absolute Percentage Error).

Cette comparaison permet d'identifier le modèle offrant le meilleur compromis entre précision des prédictions et capacité de généralisation.

In [43]:
comparaison = pd.DataFrame({
    "Modèle": [
        "Régression linéaire",
        "Ridge",
        "Lasso",
        "Arbre de décision",
        "Random Forest",
        "XGBoost",
        "LightGBM"
    ],
    "MAE": [
        mae_lin,
        mae_ridge,
        mae_lasso,
        mae_tree,
        mae_rf,
        mae_xgb_sans_conso,
        mae_lgbm_sans_conso
    ],
    "RMSE": [
        rmse_lin,
        rmse_ridge,
        rmse_lasso,
        rmse_tree,
        rmse_rf,
        rmse_xgb_sans_conso,
        rmse_lgbm_sans_conso
    ],
    "R²": [
        r2_lin,
        r2_ridge,
        r2_lasso,
        r2_tree,
        r2_rf,
        r2_xgb_sans_conso,
        r2_lgbm_sans_conso
    ],
    "MAPE": [
        mape_lin,
        mape_ridge,
        mape_lasso,
        mape_tree,
        mape_rf,
        mape_xgb_sans_conso,
        mape_lgbm_sans_conso
    ]
})

comparaison = comparaison.round({
    "MAE": 2,
    "RMSE": 2,
    "R²": 4,
    "MAPE": 4
})

comparaison

,Modèle,MAE,RMSE,R²,MAPE
0,Régression linéaire,9.63,12.92,0.8570,0.0496
1,Ridge,9.67,12.91,0.8572,0.0498
2,Lasso,12.37,16.99,0.7526,0.0639
3,Arbre de décision,3.19,5.40,0.9750,0.0173
4,Random Forest,3.17,4.82,0.9801,0.0170
5,XGBoost,3.34,4.82,0.9801,0.0181
6,LightGBM,3.71,5.44,0.9747,0.0201


### Analyse comparative des modèles

La suppression de `conso_mixte` entraîne une diminution des performances des sept modèles évalués, mais avec une intensité variable.

Les régressions linéaire et Ridge conservent des performances satisfaisantes, avec des coefficients de détermination respectifs de **0,8570** et **0,8572**, montrant que les autres caractéristiques techniques restent informatives.

Le modèle Lasso est le plus pénalisé (**R² = 0,7526**). Sa régularisation L1 peut également réduire l'influence de certaines variables explicatives, ce qui contribue potentiellement à ses performances inférieures.

L'Arbre de décision conserve d'excellentes performances (**R² = 0,9750 ; RMSE = 5,40 g/km**). Il exploite efficacement les variables restantes, notamment la masse, la puissance et le type de carburant, ce qui lui permet de maintenir un excellent niveau de prédiction malgré l'absence de `conso_mixte`.

Le **Random Forest** obtient les meilleures performances globales parmi les sept modèles évalués, avec un **R² de 0,9801**, un **RMSE de 4,82 g/km** et une **MAE de 3,17 g/km**. Il améliore notamment les résultats de l'Arbre de décision, confirmant l'intérêt de combiner plusieurs arbres pour améliorer la précision des prédictions.

**XGBoost** présente des performances très proches de celles du Random Forest, avec un **R² de 0,9801** et un **RMSE de 4,82 g/km**. Sa MAE légèrement supérieure (**3,34 g/km**) place toutefois le Random Forest devant sur cet indicateur.

Enfin, **LightGBM** conserve également une bonne capacité prédictive (**R² = 0,9747 ; RMSE = 5,44 g/km**), mais ses performances restent légèrement inférieures à celles du Random Forest et de XGBoost.

Dans l'ensemble, les modèles non linéaires, particulièrement les méthodes d'ensemble, se montrent plus performants que les modèles linéaires dans cette expérimentation.

**Le Random Forest apparaît ainsi comme le modèle le plus performant sans `conso_mixte`**, avec XGBoost très proche. Ces résultats montrent que les autres caractéristiques techniques des véhicules permettent de conserver une excellente capacité de prédiction des émissions de CO₂, même en l'absence de la consommation mixte.

## Conclusion générale de la modélisation

Cette expérimentation avait pour objectif d'évaluer l'influence de la variable `conso_mixte` sur les performances des différents modèles de régression.

La suppression de cette variable entraîne une diminution des performances de l'ensemble des modèles évalués, confirmant que la consommation mixte constitue un prédicteur majeur des émissions de CO₂.

Les modèles de Régression linéaire et Ridge conservent des performances satisfaisantes, avec des coefficients de détermination proches de 0,86, tandis que le modèle Lasso est davantage pénalisé.

Les modèles non linéaires conservent toutefois d'excellentes performances. L'Arbre de décision atteint un **R² de 0,9750** et un **RMSE de 5,40 g/km**, démontrant sa capacité à exploiter efficacement les autres caractéristiques techniques du jeu de données, notamment la masse du véhicule, la puissance et le type de carburant.

Parmi les sept modèles évalués, le **Random Forest obtient les meilleures performances globales**, avec un **R² de 0,9801**, un **RMSE de 4,82 g/km** et une **MAE de 3,17 g/km**. XGBoost présente des résultats très proches (**R² = 0,9801 ; RMSE = 4,82 g/km**), tandis que LightGBM conserve également de bonnes performances (**R² = 0,9747 ; RMSE = 5,44 g/km**).

Cette analyse met en évidence l'intérêt des modèles non linéaires, particulièrement des méthodes d'ensemble, et montre que les émissions de CO₂ peuvent être estimées avec une bonne précision même en l'absence de la variable `conso_mixte`.

Toutefois, la diminution des performances après sa suppression souligne la forte dépendance des prédictions à cette variable et justifie l'intérêt de cette expérimentation complémentaire.

**En conclusion, cette expérimentation confirme l'importance de `conso_mixte`, tout en démontrant que les autres caractéristiques techniques des véhicules contiennent suffisamment d'informations pour permettre aux modèles les plus performants de conserver un excellent niveau de prédiction.**